In [8]:
import json
import pandas as pd
import os
import urllib.request
import gzip
import shutil 

os.chdir(os.path.abspath(""))

In [9]:
root_path = os.path.dirname(os.path.dirname(os.getcwd()))
print("현재 root path:", root_path)

현재 root path: d:\Repositories\chemical_properties


In [ ]:
def load_data(filename):
    folder_zip = "/isonet/data/test_dataset_zip/"
    folder = "/isonet/data/test_dataset/"

    # filename = "Compound_000000001_000500000.sdf.gz"

    filepath_zip = root_path + folder_zip + filename 
    filepath = root_path + folder + filename[:-3]  # .gz제거한 경로

    url = "https://ftp.ncbi.nlm.nih.gov/pubchem/Compound/CURRENT-Full/SDF/" + filename


    urllib.request.urlretrieve(url, filepath_zip)


    with gzip.open(filepath_zip, 'rb') as f_in:
        with open(filepath, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)

In [ ]:
load_data("Compound_001000001_001500000.sdf.gz")

('readme.html', <http.client.HTTPMessage at 0x18cb3f5b140>)

In [ ]:
df = pd.read_json(root_path + "/data/PubChem_compound_aspirin.json")
df

,cid,smiles
0,2244,CC(=O)OC1=CC=CC=C1C(=O)O
1,123131972,CCOC1=CC=C(C=C1)NC(=O)C.CC(=O)OC1=CC=CC=C1C(=O...
2,133472,CC(=O)OC1=C(C=C(C=C1)NC(=O)CCC(=O)O)C(=O)O
3,171511,CC(=O)OC1=CC=CC=C1C(=O)O.[O-2].[Mg+2]
4,156866,CCOC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C2=CC=CC=C2OC...
...,...,...
138,79668,CC(=O)OC1=CC=CC=C1C(=O)Cl
139,175675139,CC(=O)NCCCC[C@H](C(=O)O)NC(=O)C1=CC=CC=C1O
140,169441420,C1=CC=C(C(=C1)C(=O)N[C@H](CCCCN)C(=O)O)O
141,69975280,CC(=O)OC1=CC=CC=C1C(=O)O.CN1CC[C@]23[C@@H]4[C@...


In [42]:
from rdkit import Chem

def has_isomeric_smiles(smiles):
    """
    SMILES 문자열에 입체화학 정보(Isomeric SMILES)가 포함되어 있는지 확인합니다.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return "Invalid SMILES"
    
    # 1. 분자 내 카이랄 중심(Chiral Centers) 확인
    chiral_centers = Chem.FindMolChiralCenters(mol, includeUnassigned=True)
    
    # 2. 이중 결합 기하구조(E/Z, cis/trans)에 지정된 결합이 있는지 확인
    bond_stereos = [bond.GetStereo() for bond in mol.GetBonds() if bond.GetStereo() != Chem.BondStereo.STEREONONE]
    
    # 카이랄 중심이 존재하거나 이중 결합 입체화학이 지정된 경우 True 반환
    if len(chiral_centers) > 0 or len(bond_stereos) > 0:
        return True
    
    return False

cnt = 0
for smile in df["smiles"]:
    if (has_isomeric_smiles(smile)):
        cnt +=1
print('isomeric smile 갯수 :', cnt)
print('전체 smile 갯수 :', df['smiles'].__len__())
print(f'isomeric smile 비율 : {(100*cnt/df['smiles'].__len__()):.2f}%')

isomeric smile 갯수 : 64
전체 smile 갯수 : 143
isomeric smile 비율 : 44.76%
